In [1]:
from openai import OpenAI

In [3]:
import requests

In [23]:
def fetch_temperature(lat,lon):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&current=temperature_2m"
    )

    data = response.json();
    print(data);
    return data["current"]["temperature_2m"]

In [ ]:
import json
client = OpenAI()

tool_registry= [{
    "type" : "function",
    "name" : "fetch_temperature",
    "description" : "Returns the current temperature in celcius given the coordinates",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "lat" : {"type" : "number"},
            "lon" : {"type" : "number"}
        },
        "required" : ["lat", "lon"],
        "additionalProperties" : False
    },
    "strict" : True
}]

conversation = [
    {
        "role" : "user",
        "content" : "Can you check how hot it is in San Francisco"
    }
]

first_response = client.responses.create(
    model="gpt-4.1",
    input=conversation,
    tools=tool_registry
)

print(first_response.output)

[ResponseFunctionToolCall(arguments='{"lat":37.7749,"lon":-122.4194}', call_id='call_LLD7SpMr6ieOVtLHaYF24v6z', name='fetch_temperature', type='function_call', id='fc_0f5d090f9166dbbb006903edc9542c8196a2f667f33c290cd8', status='completed')]


In [25]:
tool_suggestion = first_response.output[0]
print(tool_suggestion)

ResponseFunctionToolCall(arguments='{"lat":37.7749,"lon":-122.4194}', call_id='call_LLD7SpMr6ieOVtLHaYF24v6z', name='fetch_temperature', type='function_call', id='fc_0f5d090f9166dbbb006903edc9542c8196a2f667f33c290cd8', status='completed')


In [26]:
tool_args = json.loads(tool_suggestion.arguments);
print(tool_args)

{'lat': 37.7749, 'lon': -122.4194}


In [27]:
temp_result = fetch_temperature(tool_args["lat"], tool_args["lon"]);
print(temp_result)

{'latitude': 37.763283, 'longitude': -122.41286, 'generationtime_ms': 0.01990795135498047, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 18.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2025-10-30T22:45', 'interval': 900, 'temperature_2m': 16.0}}
16.0


In [28]:
conversation.append(tool_suggestion);
conversation.append({
    "type" : "function_call_output",
    "call_id" : tool_suggestion.call_id,
    "output" : str(temp_result)
})

completion_final = client.responses.create(
    model="gpt-4.1",
    input=conversation,
    tools=tool_registry,
)
print(completion_final.output_text)

The current temperature in San Francisco is 16°C.


Do structured outputs and chat.completions.create / beta.chat.completions.parse in Integrate Tools with Agents sections under Introduction to Agentic Patterns
Link : https://www.educative.io/module/page/P1vxGOtNzNBPX5PJY/10370001/4640179653312512/6493471261982720